In [6]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer

In [7]:
with open("knowledge_base.json", "r", encoding="utf-8") as f:
    knowledge_base = json.load(f)

In [9]:
# ── 2. Load local embedding model ────────────────────────────────────────────
# First run will download ~90MB model once, then cached locally.
 
print("Loading model...")
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.\n")

Loading model...


/opt/miniconda3/envs/torch_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded.



In [10]:
# ── 3. Embed every passage (one-time indexing step) ──────────────────────────
 
texts = [entry["text"] for entry in knowledge_base]
passage_vectors = model.encode(texts, convert_to_numpy=True)  # shape: (10, 384)
 
# Keep vectors alongside metadata
index = [
    {
        "id":     entry["id"],
        "source": entry["source"],
        "text":   entry["text"],
        "vector": passage_vectors[i],
    }
    for i, entry in enumerate(knowledge_base)
]
 
print(f"Indexed {len(index)} passages.\n")
print("=" * 70)

Indexed 10 passages.



In [11]:
# ── 4. Cosine similarity — written by hand ───────────────────────────────────
 
def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    cos(θ) = (A · B) / (‖A‖ × ‖B‖)
    Returns a float in [-1, 1].  Closer to 1 → more similar meaning.
    """
    dot_product  = np.dot(a, b)
    norm_a       = np.linalg.norm(a)
    norm_b       = np.linalg.norm(b)
    return dot_product / (norm_a * norm_b)
 

In [12]:
# ── 5. Search function ────────────────────────────────────────────────────────
 
def search(query: str, top_k: int = 3) -> list[dict]:
    query_vector = model.encode([query], convert_to_numpy=True)[0]
 
    scored = []
    for entry in index:
        score = cosine_similarity(query_vector, entry["vector"])
        scored.append({**entry, "score": score})
 
    # Sort descending by score
    scored.sort(key=lambda x: x["score"], reverse=True)
    return scored[:top_k]

In [13]:
# ── 6. Test queries ───────────────────────────────────────────────────────────
 
queries = [
    "my laptop won't switch on",
    "how do I stop being billed every month?",
    "access denied error when saving a file",
    "where do I leave my car in the evening?",
]
 
for query in queries:
    print(f"\nQUERY : {query}")
    print("-" * 70)
    results = search(query, top_k=3)
    for rank, r in enumerate(results, start=1):
        print(f"  #{rank}  [{r['id']} | {r['source']}]  score={r['score']:.4f}")
        print(f"       {r['text'][:100]}...")
    print()


QUERY : my laptop won't switch on
----------------------------------------------------------------------
  #1  [kb-02 | handbook.md]  score=0.4369
       To power up a device that won't turn on, hold the power button for ten seconds, then connect the cha...
  #2  [kb-07 | it.md]  score=0.1774
       Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to ...
  #3  [kb-09 | it.md]  score=0.1581
       Company laptops back up automatically to the cloud every night at 2am while connected to the office ...


QUERY : how do I stop being billed every month?
----------------------------------------------------------------------
  #1  [kb-05 | policy.md]  score=0.5408
       To cancel your subscription, open Account Settings and choose End Plan. Cancellation takes effect at...
  #2  [kb-06 | policy.md]  score=0.3041
       Premium plan members get priority support, with a guaranteed first response within four business hou...
  #3  [kb-10 | facilitie


# ── 7. Reflection ─────────────────────────────────────────────────────────────
 
reflection = """
## Reflection: did the best match share any words with the query?
 
---
 
### Query 1 — "my laptop won't switch on"
Best match: kb-02 (score 0.4369)
"To power up a device that won't turn on, hold the power button…"
 
Shared words: only the function words "won't" and "on" — which carry no meaning on their own.
The content words that matter — "laptop", "switch on" — have zero overlap with
"device" and "power up" in the passage.
 
A keyword search would miss this entirely. The embedding bridged two synonym pairs
simultaneously ("laptop"↔"device", "switch on"↔"power up") without any explicit
synonym list. The large gap to #2 (0.44 vs 0.18) confirms the model is confident
this is the right passage, not just the least-wrong one.
 
---
 
### Query 2 — "how do I stop being billed every month?"
Best match: kb-05 (score 0.5408)
"To cancel your subscription… Cancellation takes effect at the end of the current billing period…"
 
Shared words: none that carry meaning. "being" appears in both but is a function word.
"stop", "billed", "every month" share nothing with "cancel", "subscription", "billing period".
 
This is the strongest score in the whole test (0.54) and also the biggest
vocabulary gap. The embedding captured the underlying intent — ending a recurring
charge — even though every content word in the query differs from the passage.
This is the clearest demonstration that what is encoded is *intent*, not surface form.
 
---
 
### Query 3 — "access denied error when saving a file"
Best match: kb-08 (score 0.5437)
"The error code 0x80070005 means 'access denied'. Run the application as administrator,
or ask IT to grant your account write permission to the target folder."
 
Shared words: "access denied" and "error" appear in both — so this has the most
word overlap of the four queries.
 
Yet the interesting match is the part with no overlap: "saving a file" was correctly
linked to "write permission to the target folder". The embedding understood that
saving implies a write operation. Even where surface words matched, the embedding
added meaning beyond what keyword search could capture.
 
---
 
### Query 4 — "where do I leave my car in the evening?"
Best match: kb-01 (score 0.3257)
"Employees may park in lot B after 6pm on weekdays."
 
Shared words: none at all. "car", "leave", "evening" do not appear anywhere in kb-01.
Every content word is a synonym or paraphrase:
  "leave my car"  →  "park"
  "in the evening"  →  "after 6pm"
  "where"  →  "lot B"
 
The score (0.33) is the lowest of the four — reflecting that the vocabulary gap is
the widest — yet it still ranked #1 cleanly. This is the purest example of
semantic retrieval: a hand-written dot product surfaced the right passage with
zero word overlap.
 
---
 
### What does this tell us about what the embedding captured?
 
Across all four queries the pattern is the same: the embedding did not match
words — it matched *situations*. It grouped together texts that describe the same
real-world action or problem, regardless of how that action is worded.
 
This is exactly what a bag-of-words or TF-IDF search cannot do. Those methods
compare token frequencies; they would return zero similarity between "switch on"
and "power up" because the tokens are disjoint. The embedding model was trained
on vast text where these expressions appear in the same contexts, so it learned
to place them at nearby points in vector space.
 
The retrieval step itself — the dot product we wrote in five lines of NumPy —
is trivial arithmetic. The intelligence is entirely in those learned coordinates.
That is the core lesson: build good embeddings once, and search becomes geometry.
"""


In [15]:

# ── 7. Reflection ─────────────────────────────────────────────────────────────
 
reflection = """
## Reflection: did the best match share any words with the query?
 
---
 
### Query 1 — "my laptop won't switch on"
Best match: kb-02 (score 0.4369)
"To power up a device that won't turn on, hold the power button…"
 
Shared words: only the function words "won't" and "on" — which carry no meaning on their own.
The content words that matter — "laptop", "switch on" — have zero overlap with
"device" and "power up" in the passage.
 
A keyword search would miss this entirely. The embedding bridged two synonym pairs
simultaneously ("laptop"↔"device", "switch on"↔"power up") without any explicit
synonym list. The large gap to #2 (0.44 vs 0.18) confirms the model is confident
this is the right passage, not just the least-wrong one.
 
---
 
### Query 2 — "how do I stop being billed every month?"
Best match: kb-05 (score 0.5408)
"To cancel your subscription… Cancellation takes effect at the end of the current billing period…"
 
Shared words: none that carry meaning. "being" appears in both but is a function word.
"stop", "billed", "every month" share nothing with "cancel", "subscription", "billing period".
 
This is the strongest score in the whole test (0.54) and also the biggest
vocabulary gap. The embedding captured the underlying intent — ending a recurring
charge — even though every content word in the query differs from the passage.
This is the clearest demonstration that what is encoded is *intent*, not surface form.
 
---
 
### Query 3 — "access denied error when saving a file"
Best match: kb-08 (score 0.5437)
"The error code 0x80070005 means 'access denied'. Run the application as administrator,
or ask IT to grant your account write permission to the target folder."
 
Shared words: "access denied" and "error" appear in both — so this has the most
word overlap of the four queries.
 
Yet the interesting match is the part with no overlap: "saving a file" was correctly
linked to "write permission to the target folder". The embedding understood that
saving implies a write operation. Even where surface words matched, the embedding
added meaning beyond what keyword search could capture.
 
---
 
### Query 4 — "where do I leave my car in the evening?"
Best match: kb-01 (score 0.3257)
"Employees may park in lot B after 6pm on weekdays."
 
Shared words: none at all. "car", "leave", "evening" do not appear anywhere in kb-01.
Every content word is a synonym or paraphrase:
  "leave my car"  →  "park"
  "in the evening"  →  "after 6pm"
  "where"  →  "lot B"
 
The score (0.33) is the lowest of the four — reflecting that the vocabulary gap is
the widest — yet it still ranked #1 cleanly. This is the purest example of
semantic retrieval: a hand-written dot product surfaced the right passage with
zero word overlap.
 
---
 
### What does this tell us about what the embedding captured?
 
Across all four queries the pattern is the same: the embedding did not match
words — it matched *situations*. It grouped together texts that describe the same
real-world action or problem, regardless of how that action is worded.
 
This is exactly what a bag-of-words or TF-IDF search cannot do. Those methods
compare token frequencies; they would return zero similarity between "switch on"
and "power up" because the tokens are disjoint. The embedding model was trained
on vast text where these expressions appear in the same contexts, so it learned
to place them at nearby points in vector space.
 
The retrieval step itself — the dot product we wrote in five lines of NumPy —
is trivial arithmetic. The intelligence is entirely in those learned coordinates.
That is the core lesson: build good embeddings once, and search becomes geometry.
"""


In [16]:
print(reflection)


## Reflection: did the best match share any words with the query?
 
---
 
### Query 1 — "my laptop won't switch on"
Best match: kb-02 (score 0.4369)
"To power up a device that won't turn on, hold the power button…"
 
Shared words: only the function words "won't" and "on" — which carry no meaning on their own.
The content words that matter — "laptop", "switch on" — have zero overlap with
"device" and "power up" in the passage.
 
A keyword search would miss this entirely. The embedding bridged two synonym pairs
simultaneously ("laptop"↔"device", "switch on"↔"power up") without any explicit
synonym list. The large gap to #2 (0.44 vs 0.18) confirms the model is confident
this is the right passage, not just the least-wrong one.
 
---
 
### Query 2 — "how do I stop being billed every month?"
Best match: kb-05 (score 0.5408)
"To cancel your subscription… Cancellation takes effect at the end of the current billing period…"
 
Shared words: none that carry meaning. "being" appears in both but i

In [17]:
# ── 8. Stretch: query not covered by knowledge base ──────────────────────────
 
print("=" * 70)
print("STRETCH — Out-of-scope query\n")
 
stretch_query = "what's the wifi password?"
results = search(stretch_query, top_k=1)
top = results[0]
 
print(f"Query : {stretch_query}")
print(f"Best match score: {top['score']:.4f}  [{top['id']}]")
print(f"Text  : {top['text'][:100]}...\n")
 
THRESHOLD = 0.35
if top["score"] < THRESHOLD:
    print(f"Score {top['score']:.4f} is below threshold ({THRESHOLD}).")
    print("→ System response: 'Sorry, I don't have information about that.'")
else:
    print(f"Score {top['score']:.4f} is above threshold — a match was found.")
 

STRETCH — Out-of-scope query

Query : what's the wifi password?
Best match score: 0.3186  [kb-07]
Text  : Reset your password from the login screen by clicking 'Forgot password'. A reset link is emailed to ...

Score 0.3186 is below threshold (0.35).
→ System response: 'Sorry, I don't have information about that.'



## Stretch — "what's the wifi password?"
**Best match:** kb-07 (score 0.3186) — below threshold of 0.35
> "Reset your password from the login screen…"
 
The word `password` created a surface match, but the meaning is completely different — network access vs account recovery. The score fell below every in-scope query's top score, and the threshold correctly triggered a **"I don't have information about that"** response.
 
This shows why a similarity threshold matters: without one, a RAG system always returns *something* — even when it knows nothing relevant. A low ceiling score is a signal to abstain, not guess.
 